In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
import yaml, torch
from IPython.display import display, Image as IPyImage
from google.colab import files

YOLOV5_DIR = Path("/content/yolov5")
if not YOLOV5_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/ultralytics/yolov5.git", str(YOLOV5_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(YOLOV5_DIR / "requirements.txt")], check=True)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Upload and inspect the dataset ZIP



In [ ]:
uploaded_dataset = files.upload()
zip_files = [n for n in uploaded_dataset if n.lower().endswith(".zip")]
if not zip_files:
    raise ValueError("Upload the dataset ZIP file.")

EXTRACT_DIR = Path("/content/uploaded_weapon_dataset")
shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(zip_files[0], EXTRACT_DIR)

def valid_root(p):
    return all((p / rel).is_dir() for rel in [
        "images/train", "images/val", "labels/train", "labels/val"
    ])

roots = [p for p in [EXTRACT_DIR, *EXTRACT_DIR.rglob("*")] if p.is_dir() and valid_root(p)]
if not roots:
    raise FileNotFoundError("Expected images/train, images/val, labels/train, labels/val folders. Check the ZIP structure.")
DATA_ROOT = roots[0]
print("Dataset root:", DATA_ROOT)
for split in ("train", "val"):
    imgs = [p for p in (DATA_ROOT / "images" / split).rglob("*") if p.is_file() and p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp"}]
    labels = list((DATA_ROOT / "labels" / split).glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(labels)} label files")

## 3. Upload the trained checkpoint



In [ ]:
uploaded_weights = files.upload()
pt_files = [n for n in uploaded_weights if n.lower().endswith(".pt")]
if not pt_files:
    raise ValueError("Upload the trained best.pt checkpoint.")
WEIGHTS_PATH = Path("/content") / pt_files[0]
print("Weights:", WEIGHTS_PATH, "| MB:", round(WEIGHTS_PATH.stat().st_size / 1048576, 2))

## 4. Create and check the dataset configuration


In [ ]:
DATA_YAML = Path("/content/weapon_eval.yaml")
cfg = {
    "path": str(DATA_ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "Knife", 1: "Handgun"}
}
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text())

extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
val_images = [p for p in (DATA_ROOT / "images/val").rglob("*") if p.is_file() and p.suffix.lower() in extensions]
val_labels = list((DATA_ROOT / "labels/val").glob("*.txt"))
if not val_images:
    raise RuntimeError("No validation images found in images/val.")
if not val_labels:
    raise RuntimeError("No label files found in labels/val.")
print(f"Validation set ready: {len(val_images)} images.")

## 5. Evaluate the model



In [ ]:
IMG_SIZE = 640
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
DEVICE = "0" if torch.cuda.is_available() else "cpu"

cmd = [
    sys.executable, str(YOLOV5_DIR / "val.py"),
    "--data", str(DATA_YAML),
    "--weights", str(WEIGHTS_PATH),
    "--imgsz", str(IMG_SIZE),
    "--batch-size", str(BATCH_SIZE),
    "--task", "val",
    "--device", DEVICE,
    "--workers", "2" if torch.cuda.is_available() else "0",
    "--project", "/content/evaluation_runs",
    "--name", "weapon_model_eval",
    "--exist-ok"
]

print("Running:", " ".join(cmd))

result = subprocess.run(
    cmd,
    cwd=str(YOLOV5_DIR),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError("Validation failed. Check the output above.")

## 6. View the evaluation plots



In [ ]:
EVAL_DIR = Path("/content/evaluation_runs/weapon_model_eval")
print("Output directory:", EVAL_DIR)
for name in ["confusion_matrix.png", "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png"]:
    p = EVAL_DIR / name
    if p.exists():
        print(name)
        display(IPyImage(filename=str(p)))
if EVAL_DIR.exists():
    print("All output files:")
    for p in sorted(EVAL_DIR.rglob("*")):
        if p.is_file():
            print(p)